# 02 — Shared BPE tokenization

> **Status:** implementation in progress.

- **Mapped issue:** [#3](https://github.com/majorgilles/transformer-2017-reproduction/issues/3)
- **Depends on:** `01_data_contracts_provenance.ipynb` / issue #2.


In [1]:
#| default_exp tokenization


## Stable special-token contract

The tokenizer reserves padding, unknown, beginning-of-sequence, and end-of-sequence symbols before learning ordinary BPE pieces. Their tuple order fixes IDs 0–3, so later datasets, embeddings, checkpoints, and inference code interpret the same integers consistently.


In [2]:
#| export
from __future__ import annotations

from collections.abc import Iterable
from enum import StrEnum
from math import ceil
from pathlib import Path
from statistics import fmean, median
from typing import Final, Self

from pydantic import BaseModel, ConfigDict, Field, model_validator
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

from transformer_2017_reproduction.identity import sha256_bytes


class SpecialToken(StrEnum):
    """Reserved tokens with fixed vocabulary positions."""

    PAD = "<pad>"  # Fills unused positions when sequences are batched to one length.
    UNK = "<unk>"  # Represents input text that is absent from the learned vocabulary.
    BOS = "<bos>"  # Marks the beginning of a token sequence.
    EOS = "<eos>"  # Marks the end of a token sequence.


SPECIAL_TOKENS: Final[tuple[str, ...]] = tuple(token.value for token in SpecialToken)

In [3]:
from importlib.metadata import version as package_version
from time import perf_counter

from transformer_2017_reproduction.data import (
    iter_manifest_examples,
    load_manifest,
)
from transformer_2017_reproduction.identity import sha256_file

In [4]:
assert SPECIAL_TOKENS == ("<pad>", "<unk>", "<bos>", "<eos>")
assert len(SPECIAL_TOKENS) == len(set(SPECIAL_TOKENS))

## Validated training configuration

A tokenizer artifact is meaningful only when its training settings are known. This frozen Pydantic model rejects unknown fields, impossible vocabulary sizes, and reordered special tokens so invalid settings fail before training begins.


In [5]:
#| export
class BPETrainingConfig(BaseModel):
    """Validated settings that determine a learned BPE vocabulary."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    vocab_size: int = Field(gt=len(SPECIAL_TOKENS))  # Maximum learned vocabulary size.
    min_frequency: int = Field(ge=1)  # Minimum corpus count required for a merge.
    special_tokens: tuple[str, ...] = SPECIAL_TOKENS  # Fixed values and ID order.

    @model_validator(mode="after")
    def require_canonical_special_tokens(self) -> Self:
        if self.special_tokens != SPECIAL_TOKENS:
            raise ValueError("special tokens must use the canonical values and order")
        return self

In [6]:
# Small test configuration; this is not the canonical WMT vocabulary size.
fixture_config = BPETrainingConfig(vocab_size=46, min_frequency=2)

assert fixture_config.special_tokens == SPECIAL_TOKENS
assert fixture_config.model_dump() == {
    "vocab_size": 46,
    "min_frequency": 2,
    "special_tokens": ("<pad>", "<unk>", "<bos>", "<eos>"),
}

## Shared BPE training pipeline

Both English and German strings flow through one trainer, producing one shared token-to-ID space. Before BPE sees the text, `Metaspace` replaces whitespace with one visible marker character. `replacement` selects that marker, `prepend_scheme="always"` also places it before the first word, and `split=True` tells the pre-tokenizer to split at marker positions. The matching Metaspace decoder uses the same marker and prepend scheme to restore readable spaces.


In [7]:
#| export
def train_bpe(texts: Iterable[str], config: BPETrainingConfig) -> Tokenizer:
    """Train a shared BPE tokenizer from an iterable of text."""

    # The model learns merge rules and falls back to <unk> for unseen symbols.
    tokenizer = Tokenizer(models.BPE(unk_token=SpecialToken.UNK.value))
    # Metaspace makes whitespace visible to BPE instead of discarding word boundaries.
    tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(
        # Replace each whitespace boundary with the conventional ▁ marker.
        replacement="▁",
        # Add the marker before the first word as well as words following spaces.
        prepend_scheme="always",
        # Split at marker positions so boundaries become explicit pre-tokens.
        split=True,
    )

    # The trainer inserts special tokens first, preserving their canonical IDs.
    trainer = trainers.BpeTrainer(
        vocab_size=config.vocab_size,
        min_frequency=config.min_frequency,
        special_tokens=list(config.special_tokens),
        show_progress=False,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)

    # The decoder reverses the Metaspace transformation after token IDs are decoded.
    tokenizer.decoder = decoders.Metaspace(
        # Interpret the same ▁ marker as whitespace; this must match the pre-tokenizer.
        replacement="▁",
        # Match the synthetic marker placed before the first word.
        prepend_scheme="always",
    )
    return tokenizer

## Deterministic bilingual fixture

This bounded English-German corpus contains related word forms so the learned vocabulary exhibits a realistic mixture of whole frequent words and reusable subword pieces. A vocabulary limit of 46 and minimum frequency of 2 are fixture-only settings chosen to make those merges visible; they are not the canonical WMT choices.


In [8]:
fixture_texts = (
    "the cat sleeps",
    "the cats sleep",
    "the cat slept",
    "a sleeping cat",
    "die katze schläft",
    "die katzen schlafen",
    "die katze schlief",
    "eine schlafende katze",
)

fixture_tokenizer = train_bpe(fixture_texts, fixture_config)

for expected_id, token in enumerate(SPECIAL_TOKENS):
    assert fixture_tokenizer.token_to_id(token) == expected_id

## Visible result: one vocabulary for both languages

The result below encodes both fixture sentences and new combinations with the same vocabulary. Frequent forms can become whole tokens, while related forms split into shared stems and endings such as `▁cat` + `s` or `▁schlaf` + `en`. The `▁` character is the conventional visible space marker used internally by Metaspace.


In [9]:
demonstration_texts = (
    "the cats sleep",
    "die katzen schlafen",
    "the cat is sleeping",
    "die katze kann schlafen",
)

print(f"Learned shared vocabulary size: {fixture_tokenizer.get_vocab_size()}")
print("Boundary marker: ▁ means a space or the start of a sentence.")
for text in demonstration_texts:
    encoded = fixture_tokenizer.encode(text)
    decoded = fixture_tokenizer.decode(encoded.ids)
    print(f"{text!r} -> {encoded.tokens} -> {encoded.ids} -> {decoded!r}")

Learned shared vocabulary size: 46
Boundary marker: ▁ means a space or the start of a sentence.
'the cats sleep' -> ['▁the', '▁cat', 's', '▁sleep'] -> [40, 29, 16, 41] -> 'the cats sleep'
'die katzen schlafen' -> ['▁die', '▁katze', 'n', '▁schlaf', 'en'] -> [39, 33, 14, 45, 43] -> 'die katzen schlafen'
'the cat is sleeping' -> ['▁the', '▁cat', '▁', 'i', 's', '▁sleep', 'in', 'g'] -> [40, 29, 20, 11, 16, 41, 44, 9] -> 'the cat is sleeping'
'die katze kann schlafen' -> ['▁die', '▁katze', '▁', 'k', 'a', 'n', 'n', '▁schlaf', 'en'] -> [39, 33, 20, 12, 4, 14, 14, 45, 43] -> 'die katze kann schlafen'


## Fixture vocabulary-size and sequence-inflation comparison

A smaller vocabulary preserves more character-level pieces, while the selected fixture vocabulary spends additional merge slots to shorten sequences. Comparing both on the same bilingual fixture isolates that trade-off without touching development or final-test text.

In [10]:
comparison_vocab_sizes = (36, fixture_config.vocab_size)
comparison_rows: list[tuple[int, int, float]] = []
fixture_whitespace_words = sum(len(text.split()) for text in fixture_texts)

for vocabulary_size in comparison_vocab_sizes:
    candidate = train_bpe(
        fixture_texts,
        BPETrainingConfig(vocab_size=vocabulary_size, min_frequency=2),
    )
    candidate_token_count = sum(len(candidate.encode(text).ids) for text in fixture_texts)
    comparison_rows.append(
        (
            candidate.get_vocab_size(),
            candidate_token_count,
            candidate_token_count / fixture_whitespace_words,
        )
    )

assert comparison_rows == [(36, 70, 70 / 24), (46, 44, 44 / 24)]
assert comparison_rows[1][1] < comparison_rows[0][1]

print("Fixture vocabulary/sequence comparison:")
print("  vocabulary | subword tokens | tokens/whitespace word")
for vocabulary_size, token_count, inflation in comparison_rows:
    print(f"  {vocabulary_size:>10} | {token_count:>14} | {inflation:.3f}")
print("  selected fixture vocabulary: 46 (shorter sequences with reusable subwords)")

Fixture vocabulary/sequence comparison:
  vocabulary | subword tokens | tokens/whitespace word
          36 |             70 | 2.917
          46 |             44 | 1.833
  selected fixture vocabulary: 46 (shorter sequences with reusable subwords)


## Visible round-trip evidence

Each example is encoded into shared-vocabulary pieces and decoded back into text.
The printed rows make the behavior inspectable, while the checks fail execution if
text changes or the tokenizer unexpectedly uses `<unk>`.

In [11]:
round_trip_texts = fixture_texts + demonstration_texts

for text in round_trip_texts:
    encoded = fixture_tokenizer.encode(text)
    decoded = fixture_tokenizer.decode(encoded.ids)

    assert decoded == text
    assert SpecialToken.UNK.value not in encoded.tokens

print(
    f"Round-trip check passed for {len(round_trip_texts)} English/German examples; "
    "no <unk> tokens were used."
)
print("Boundary marker: ▁ means a space or the start of a sentence.")
print("\nRepresentative tokenizations:")

for text in demonstration_texts:
    encoded = fixture_tokenizer.encode(text)
    decoded = fixture_tokenizer.decode(encoded.ids)

    print(f"{text!r}\n  tokens={encoded.tokens}\n  ids={encoded.ids}\n  decoded={decoded!r}")

Round-trip check passed for 12 English/German examples; no <unk> tokens were used.
Boundary marker: ▁ means a space or the start of a sentence.

Representative tokenizations:
'the cats sleep'
  tokens=['▁the', '▁cat', 's', '▁sleep']
  ids=[40, 29, 16, 41]
  decoded='the cats sleep'
'die katzen schlafen'
  tokens=['▁die', '▁katze', 'n', '▁schlaf', 'en']
  ids=[39, 33, 14, 45, 43]
  decoded='die katzen schlafen'
'the cat is sleeping'
  tokens=['▁the', '▁cat', '▁', 'i', 's', '▁sleep', 'in', 'g']
  ids=[40, 29, 20, 11, 16, 41, 44, 9]
  decoded='the cat is sleeping'
'die katze kann schlafen'
  tokens=['▁die', '▁katze', '▁', 'k', 'a', 'n', 'n', '▁schlaf', 'en']
  ids=[39, 33, 20, 12, 4, 14, 14, 45, 43]
  decoded='die katze kann schlafen'


## Stable tokenizer serialization and identity

`Tokenizer.to_str(pretty=False)` serializes the learned vocabulary, ordered BPE merges, Metaspace rules, decoder rules, and special-token settings as compact JSON. Encoding that JSON as UTF-8 creates the exact bytes stored in the artifact. Training the same fixture twice must produce identical bytes, while SHA-256 gives those bytes a compact identity that changes if any serialized tokenizer state changes.


In [12]:
#| export
def tokenizer_as_bytes(tokenizer: Tokenizer) -> bytes:
    """Serialize a trained tokenizer into deterministic UTF-8 bytes."""

    # Compact JSON removes formatting differences from the artifact identity.
    return tokenizer.to_str(pretty=False).encode("utf-8")


#| export
def tokenizer_sha256(tokenizer: Tokenizer) -> str:
    """Return the SHA-256 identity of a trained tokenizer."""

    # Hash the complete serialized state, not only the requested training configuration.
    return sha256_bytes(tokenizer_as_bytes(tokenizer))

In [13]:
retrained_fixture_tokenizer = train_bpe(fixture_texts, fixture_config)

fixture_tokenizer_bytes = tokenizer_as_bytes(fixture_tokenizer)
retrained_tokenizer_bytes = tokenizer_as_bytes(retrained_fixture_tokenizer)
deterministic_retraining = fixture_tokenizer_bytes == retrained_tokenizer_bytes
fixture_tokenizer_digest = tokenizer_sha256(fixture_tokenizer)

assert deterministic_retraining
assert len(fixture_tokenizer_digest) == 64

print("Tokenizer artifact identity:")
print(f"  serialized bytes: {len(fixture_tokenizer_bytes)}")
print(f"  SHA-256: {fixture_tokenizer_digest}")
print(f"  deterministic retraining: {deterministic_retraining}")

Tokenizer artifact identity:
  serialized bytes: 1650
  SHA-256: c56e04377b8ef8d97a0f379a9e622d9c610e3c80b6e5acfecfbe09c6120aa697
  deterministic retraining: True


## Validated tokenizer loading

Serialized tokenizer bytes cross a trust boundary when they are loaded from storage. The caller supplies the expected SHA-256 recorded by trusted metadata; the loader hashes the received `content` and compares the two values before parsing JSON. A mismatch raises `ValueError`, so altered or accidentally substituted vocabulary data never reaches encoding or decoding.


In [14]:
#| export
def tokenizer_from_bytes(content: bytes, expected_sha256: str) -> Tokenizer:
    """Verify serialized tokenizer bytes and reconstruct the tokenizer."""

    # Verify raw bytes before decoding or asking the tokenizer library to parse them.
    actual_sha256 = sha256_bytes(content)
    if actual_sha256 != expected_sha256:
        raise ValueError(
            f"tokenizer SHA-256 mismatch: expected {expected_sha256}, got {actual_sha256}"
        )

    return Tokenizer.from_str(content.decode("utf-8"))

In [15]:
loaded_fixture_tokenizer = tokenizer_from_bytes(
    fixture_tokenizer_bytes,
    fixture_tokenizer_digest,
)

example_text = "die katzen schlafen"
loaded_encoding = loaded_fixture_tokenizer.encode(example_text)
loaded_decoding = loaded_fixture_tokenizer.decode(loaded_encoding.ids)

assert loaded_decoding == example_text

try:
    tokenizer_from_bytes(
        fixture_tokenizer_bytes,
        expected_sha256="0" * 64,
    )
except ValueError as error:
    mismatch_message = str(error)
else:
    raise AssertionError("an incorrect tokenizer identity was accepted")

print("Validated tokenizer loading:")
print(f"  expected identity accepted: {loaded_decoding!r}")
print(f"  incorrect identity rejected: {mismatch_message}")

Validated tokenizer loading:
  expected identity accepted: 'die katzen schlafen'
  incorrect identity rejected: tokenizer SHA-256 mismatch: expected 0000000000000000000000000000000000000000000000000000000000000000, got c56e04377b8ef8d97a0f379a9e622d9c610e3c80b6e5acfecfbe09c6120aa697


## Reproducible fixture tokenizer artifact

The trained fixture tokenizer is written below the repository-root `artifacts/` directory, which is intentionally ignored by Git and can be regenerated by executing this notebook. The writer first serializes the complete tokenizer, accepts an existing file only when its bytes are identical, and otherwise writes a temporary `.part` file before atomic replacement. This prevents both accidental identity changes and partially written artifacts after interruption.


In [16]:
#| export
def write_tokenizer_artifact(path: Path, tokenizer: Tokenizer) -> str:
    """Atomically write immutable tokenizer bytes and return their SHA-256 identity."""

    # Compute the complete content before touching the destination path.
    content = tokenizer_as_bytes(tokenizer)
    path.parent.mkdir(parents=True, exist_ok=True)

    # Treat an existing path as immutable: identical bytes are reusable, others fail.
    if path.exists():
        if path.read_bytes() != content:
            raise FileExistsError(
                f"refusing to replace tokenizer artifact with different content: {path}"
            )
    else:
        # Publish only complete bytes: the temporary file is atomically renamed last.
        part_path = path.with_suffix(path.suffix + ".part")
        part_path.write_bytes(content)
        part_path.replace(path)

    return sha256_bytes(content)

In [17]:
def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing the project marker."""

    # Jupyter may run from the repository root or from notebooks/, so search upward.
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate

    raise FileNotFoundError("could not find project root containing pyproject.toml")


project_root = find_project_root(Path.cwd())
fixture_artifact_path = project_root / "artifacts" / "tokenizers" / "fixture-shared-bpe.json"
written_fixture_digest = write_tokenizer_artifact(
    fixture_artifact_path,
    fixture_tokenizer,
)

loaded_artifact_tokenizer = tokenizer_from_bytes(
    fixture_artifact_path.read_bytes(),
    written_fixture_digest,
)
artifact_example = "the cats sleep"
artifact_round_trip = loaded_artifact_tokenizer.decode(
    loaded_artifact_tokenizer.encode(artifact_example).ids
)

assert written_fixture_digest == fixture_tokenizer_digest
assert artifact_round_trip == artifact_example

print("Fixture tokenizer artifact:")
print(f"  path: {fixture_artifact_path.relative_to(project_root).as_posix()}")
print(f"  byte count: {fixture_artifact_path.stat().st_size}")
print(f"  SHA-256: {written_fixture_digest}")
print(f"  loaded round trip: {artifact_round_trip!r}")

Fixture tokenizer artifact:
  path: artifacts/tokenizers/fixture-shared-bpe.json
  byte count: 1650
  SHA-256: c56e04377b8ef8d97a0f379a9e622d9c610e3c80b6e5acfecfbe09c6120aa697
  loaded round trip: 'the cats sleep'


## Structured tokenizer artifact identity

The tokenizer bytes are only reproducible when their configuration, source manifest, library version, and accepted limitations remain attached to the digest. This strict model makes those trust-boundary fields explicit.

In [18]:
#| export
class TokenizerArtifactMetadata(BaseModel):
    """Identity and reproduction context for a frozen tokenizer artifact."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    schema_version: int
    relative_path: str
    tokenizer_sha256: str
    dataset_manifest_sha256: str
    training_config: BPETrainingConfig
    tokenizer_library: str
    tokenizer_library_version: str
    training_pair_count: int
    accepted_limitations: tuple[str, ...]

### Visible result: metadata binds bytes to their recipe

A fixture record demonstrates the strict serialized shape before the full WMT identity is recorded below.

In [19]:
fixture_metadata = TokenizerArtifactMetadata(
    schema_version=1,
    relative_path="artifacts/tokenizers/fixture-shared-bpe.json",
    tokenizer_sha256=tokenizer_sha256(fixture_tokenizer),
    dataset_manifest_sha256=sha256_bytes("\n".join(fixture_texts).encode("utf-8")),
    training_config=fixture_config,
    tokenizer_library="tokenizers",
    tokenizer_library_version=package_version("tokenizers"),
    training_pair_count=len(fixture_texts) // 2,
    accepted_limitations=(),
)

fixture_metadata_json = fixture_metadata.model_dump_json(indent=2)
assert TokenizerArtifactMetadata.model_validate_json(fixture_metadata_json) == fixture_metadata

print("Fixture artifact metadata:")
print(fixture_metadata_json)

Fixture artifact metadata:
{
  "schema_version": 1,
  "relative_path": "artifacts/tokenizers/fixture-shared-bpe.json",
  "tokenizer_sha256": "c56e04377b8ef8d97a0f379a9e622d9c610e3c80b6e5acfecfbe09c6120aa697",
  "dataset_manifest_sha256": "5af912cb53ced5430b50252ae39bbee0b42d23bb5a1a83eaa3a53c076b3ff33a",
  "training_config": {
    "vocab_size": 46,
    "min_frequency": 2,
    "special_tokens": [
      "<pad>",
      "<unk>",
      "<bos>",
      "<eos>"
    ]
  },
  "tokenizer_library": "tokenizers",
  "tokenizer_library_version": "0.23.1",
  "training_pair_count": 4,
  "accepted_limitations": []
}


## Real WMT training-path smoke run

Before the full 37k-vocabulary run, this bounded smoke run verifies the complete
real-data path: validated manifest, verified training shards, shared English-German
iteration, BPE training, and round-trip encoding. Its tokenizer is temporary and must
not be confused with the final full-corpus artifact.

In [20]:
from itertools import islice

from transformer_2017_reproduction.data import iter_bpe_training_text

wmt_root = project_root / "data" / "wmt14_en_de"
wmt_manifest_path = wmt_root / "manifests" / "wmt14-en-de-shard-100000.json"
wmt_manifest = load_manifest(wmt_manifest_path)
wmt_manifest_digest = sha256_file(wmt_manifest_path)
train_pair_count = sum(
    record.example_count for record in wmt_manifest.shards if record.split == "train"
)

smoke_run_started = perf_counter()
smoke_text_count = 20_000
smoke_texts = tuple(
    islice(
        iter_bpe_training_text(wmt_root, wmt_manifest),
        smoke_text_count,
    )
)
smoke_data_seconds = perf_counter() - smoke_run_started

smoke_config = BPETrainingConfig(
    vocab_size=2_000,
    min_frequency=2,
)
smoke_training_started = perf_counter()
smoke_tokenizer = train_bpe(smoke_texts, smoke_config)
smoke_training_seconds = perf_counter() - smoke_training_started
smoke_total_seconds = perf_counter() - smoke_run_started

smoke_examples = smoke_texts[:2]
for text in smoke_examples:
    encoded = smoke_tokenizer.encode(text)
    assert smoke_tokenizer.decode(encoded.ids) == text

print("Real WMT tokenizer smoke run:")
print(f"  manifest SHA-256: {wmt_manifest_digest}")
print(f"  approved training pairs: {train_pair_count:,}")
print(f"  development pairs excluded: {wmt_manifest.total_examples - train_pair_count:,}")
print(f"  training strings consumed: {len(smoke_texts):,}")
print(f"  learned vocabulary size: {smoke_tokenizer.get_vocab_size():,}")
print(f"  shard loading/verification seconds: {smoke_data_seconds:.2f}")
print(f"  BPE fitting seconds: {smoke_training_seconds:.2f}")
print(f"  total smoke-run seconds: {smoke_total_seconds:.2f}")

for text in smoke_examples:
    encoded = smoke_tokenizer.encode(text)
    print(f"  input: {text!r}")
    print(f"  tokens: {encoded.tokens}")
    print(f"  round trip: {smoke_tokenizer.decode(encoded.ids)!r}")
    print("  " + "-" * 20)

Real WMT tokenizer smoke run:
  manifest SHA-256: 7b101e47d2e756fff61b8cb9a3d9c66228f148b02b7bb0d1d9047df9176fa66f
  approved training pairs: 4,508,785
  development pairs excluded: 3,000
  training strings consumed: 20,000
  learned vocabulary size: 2,000
  shard loading/verification seconds: 0.32
  BPE fitting seconds: 0.41
  total smoke-run seconds: 0.73
  input: 'Resumption of the session'
  tokens: ['▁R', 'es', 'um', 'pt', 'ion', '▁of', '▁the', '▁s', 'ess', 'ion']
  round trip: 'Resumption of the session'
  --------------------
  input: 'Wiederaufnahme der Sitzungsperiode'
  tokens: ['▁Wie', 'der', 'auf', 'nahme', '▁der', '▁S', 'itz', 'ung', 'sp', 'er', 'io', 'de']
  round trip: 'Wiederaufnahme der Sitzungsperiode'
  --------------------


## Full-corpus shared BPE training

This run trains the paper-first 37k shared vocabulary from every approved English and
German training sentence. It streams verified training shards without materializing the
complete corpus as a Python tuple. Development and final-test text remain excluded.
The resulting artifact becomes the candidate canonical tokenizer evaluated below.

In [21]:
CANONICAL_TOKENIZER_SHA256: Final[str] = (
    "2826114029ded109107969c689cd62307e128dff84ad661211a26e4dfd272f74"
)
canonical_config = BPETrainingConfig(
    vocab_size=37_000,
    min_frequency=2,
)
canonical_artifact_path = (
    project_root / "artifacts" / "tokenizers" / "wmt14-en-de-shared-bpe-37000.json"
)
canonical_training_seconds: float | None

if canonical_artifact_path.exists():
    canonical_bytes = canonical_artifact_path.read_bytes()
    canonical_tokenizer = tokenizer_from_bytes(
        canonical_bytes,
        CANONICAL_TOKENIZER_SHA256,
    )
    canonical_digest = CANONICAL_TOKENIZER_SHA256
    canonical_action = "loaded existing verified artifact"
    canonical_training_seconds = None
else:
    print(
        f"Training shared BPE from {train_pair_count:,} pairs "
        f"({train_pair_count * 2:,} English/German strings)...",
        flush=True,
    )

    canonical_started = perf_counter()
    canonical_tokenizer = train_bpe(
        iter_bpe_training_text(wmt_root, wmt_manifest),
        canonical_config,
    )
    canonical_training_seconds = perf_counter() - canonical_started

    canonical_digest = write_tokenizer_artifact(
        canonical_artifact_path,
        canonical_tokenizer,
    )
    if canonical_digest != CANONICAL_TOKENIZER_SHA256:
        raise ValueError(
            "retrained canonical tokenizer identity mismatch: "
            f"expected {CANONICAL_TOKENIZER_SHA256}, got {canonical_digest}"
        )

    canonical_bytes = tokenizer_as_bytes(canonical_tokenizer)
    canonical_action = "trained and wrote verified artifact"

for expected_id, token in enumerate(SPECIAL_TOKENS):
    assert canonical_tokenizer.token_to_id(token) == expected_id

for text in smoke_examples:
    encoded = canonical_tokenizer.encode(text)
    assert canonical_tokenizer.decode(encoded.ids) == text
    assert SpecialToken.UNK.value not in encoded.tokens

print("Full-corpus tokenizer artifact:")
print(f"  training pairs: {train_pair_count:,}")
print(f"  training strings: {train_pair_count * 2:,}")
print(f"  vocabulary size: {canonical_tokenizer.get_vocab_size():,}")
print(f"  action: {canonical_action}")
if canonical_training_seconds is not None:
    print(f"  training seconds: {canonical_training_seconds:.2f}")
print(f"  artifact bytes: {len(canonical_bytes):,}")
print(f"  path: {canonical_artifact_path.relative_to(project_root).as_posix()}")
print(f"  tokenizer SHA-256: {canonical_digest}")
print(f"  dataset manifest SHA-256: {wmt_manifest_digest}")

for text in smoke_examples:
    encoded = canonical_tokenizer.encode(text)
    print(f"  input: {text!r}")
    print(f"  tokens: {encoded.tokens}")
    print(f"  round trip: {canonical_tokenizer.decode(encoded.ids)!r}")

Full-corpus tokenizer artifact:
  training pairs: 4,508,785
  training strings: 9,017,570
  vocabulary size: 37,000
  action: loaded existing verified artifact
  artifact bytes: 1,160,784
  path: artifacts/tokenizers/wmt14-en-de-shared-bpe-37000.json
  tokenizer SHA-256: 2826114029ded109107969c689cd62307e128dff84ad661211a26e4dfd272f74
  dataset manifest SHA-256: 7b101e47d2e756fff61b8cb9a3d9c66228f148b02b7bb0d1d9047df9176fa66f
  input: 'Resumption of the session'
  tokens: ['▁Res', 'um', 'ption', '▁of', '▁the', '▁session']
  round trip: 'Resumption of the session'
  input: 'Wiederaufnahme der Sitzungsperiode'
  tokens: ['▁Wiederaufnahme', '▁der', '▁Sitzungsperiode']
  round trip: 'Wiederaufnahme der Sitzungsperiode'


## Development vocabulary and sequence-length report

The 3,000 held-out development pairs did not influence BPE training. This report measures English, German, and combined token lengths, unknown-token use, and development vocabulary utilization. Whitespace words provide a simple baseline for showing how many subword positions the Transformer will process.


In [22]:
#| export
class TokenizationReport(BaseModel):
    """Aggregate tokenizer behavior over a bounded text collection."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    text_count: int
    whitespace_word_count: int
    token_count: int
    unknown_token_count: int
    unique_token_count: int
    mean_tokens_per_text: float
    median_tokens_per_text: float
    p95_tokens_per_text: int
    max_tokens_per_text: int
    tokens_per_whitespace_word: float
    vocabulary_utilization_fraction: float


def _nearest_rank_percentile(values: list[int], percentile: int) -> int:
    """Return a deterministic nearest-rank percentile."""

    if not values:
        raise ValueError("cannot calculate a percentile from no values")
    if percentile < 1 or percentile > 100:
        raise ValueError("percentile must be between 1 and 100")

    ordered = sorted(values)
    rank = ceil((percentile / 100) * len(ordered))
    return ordered[rank - 1]


def summarize_tokenization(
    tokenizer: Tokenizer,
    texts: Iterable[str],
) -> TokenizationReport:
    """Summarize token lengths, unknowns, and vocabulary use."""

    unknown_id = tokenizer.token_to_id(SpecialToken.UNK.value)
    if unknown_id is None:
        raise ValueError("tokenizer does not contain the canonical unknown token")

    text_count = 0
    whitespace_word_count = 0
    token_count = 0
    unknown_token_count = 0
    token_lengths: list[int] = []
    used_token_ids: set[int] = set()

    for text in texts:
        encoded = tokenizer.encode(text)
        length = len(encoded.ids)

        text_count += 1
        whitespace_word_count += len(text.split())
        token_count += length
        unknown_token_count += encoded.ids.count(unknown_id)
        token_lengths.append(length)
        used_token_ids.update(encoded.ids)

    if text_count == 0:
        raise ValueError("cannot summarize an empty text collection")
    if whitespace_word_count == 0:
        raise ValueError("cannot compare against zero whitespace words")

    return TokenizationReport(
        text_count=text_count,
        whitespace_word_count=whitespace_word_count,
        token_count=token_count,
        unknown_token_count=unknown_token_count,
        unique_token_count=len(used_token_ids),
        mean_tokens_per_text=fmean(token_lengths),
        median_tokens_per_text=float(median(token_lengths)),
        p95_tokens_per_text=_nearest_rank_percentile(token_lengths, 95),
        max_tokens_per_text=max(token_lengths),
        tokens_per_whitespace_word=token_count / whitespace_word_count,
        vocabulary_utilization_fraction=(len(used_token_ids) / tokenizer.get_vocab_size()),
    )

In [23]:
development_examples = tuple(
    iter_manifest_examples(
        wmt_root,
        wmt_manifest,
        "development",
    )
)

source_report = summarize_tokenization(
    canonical_tokenizer,
    (example.source_text for example in development_examples),
)
target_report = summarize_tokenization(
    canonical_tokenizer,
    (example.target_text for example in development_examples),
)
combined_report = summarize_tokenization(
    canonical_tokenizer,
    (
        text
        for example in development_examples
        for text in (example.source_text, example.target_text)
    ),
)

assert source_report.text_count == 3_000
assert target_report.text_count == 3_000
assert combined_report.text_count == 6_000


def print_tokenization_report(
    label: str,
    report: TokenizationReport,
) -> None:
    print(f"{label}:")
    print(f"  texts: {report.text_count:,}")
    print(f"  tokens: {report.token_count:,}")
    print(f"  mean tokens/text: {report.mean_tokens_per_text:.2f}")
    print(f"  median tokens/text: {report.median_tokens_per_text:.2f}")
    print(f"  p95 tokens/text: {report.p95_tokens_per_text}")
    print(f"  max tokens/text: {report.max_tokens_per_text}")
    print(f"  tokens/whitespace word: {report.tokens_per_whitespace_word:.3f}")
    print(f"  unknown tokens: {report.unknown_token_count:,}")
    print(f"  development vocabulary utilization: {report.vocabulary_utilization_fraction:.1%}")


print("Development tokenization evidence:")
print_tokenization_report("English source", source_report)
print_tokenization_report("German target", target_report)
print_tokenization_report("Combined", combined_report)

Development tokenization evidence:
English source:
  texts: 3,000
  tokens: 72,029
  mean tokens/text: 24.01
  median tokens/text: 21.00
  p95 tokens/text: 52
  max tokens/text: 120
  tokens/whitespace word: 1.284
  unknown tokens: 1
  development vocabulary utilization: 27.3%
German target:
  texts: 3,000
  tokens: 77,509
  mean tokens/text: 25.84
  median tokens/text: 23.00
  p95 tokens/text: 56
  max tokens/text: 132
  tokens/whitespace word: 1.417
  unknown tokens: 0
  development vocabulary utilization: 32.5%
Combined:
  texts: 6,000
  tokens: 149,538
  mean tokens/text: 24.92
  median tokens/text: 22.00
  p95 tokens/text: 54
  max tokens/text: 132
  tokens/whitespace word: 1.350
  unknown tokens: 1
  development vocabulary utilization: 51.4%


## Accepted canonical tokenizer decision

The 37,000-token shared vocabulary is frozen as the paper-first baseline. On the untouched development split it produces 149,538 tokens across 6,000 texts, uses 51.4% of the vocabulary, and encounters one English `<unk>` token. That single occurrence is accepted as a bounded limitation; byte fallback and final-test inspection remain out of scope.

In [24]:
canonical_metadata = TokenizerArtifactMetadata(
    schema_version=1,
    relative_path=canonical_artifact_path.relative_to(project_root).as_posix(),
    tokenizer_sha256=canonical_digest,
    dataset_manifest_sha256=wmt_manifest_digest,
    training_config=canonical_config,
    tokenizer_library="tokenizers",
    tokenizer_library_version=package_version("tokenizers"),
    training_pair_count=train_pair_count,
    accepted_limitations=(
        "One <unk> token occurs across the 3,000 English development sentences.",
    ),
)

assert canonical_tokenizer.get_vocab_size() == 37_000
assert package_version("tokenizers") == "0.23.1"
assert source_report.unknown_token_count == 1
assert target_report.unknown_token_count == 0
assert combined_report.token_count == 149_538
assert canonical_metadata.tokenizer_sha256 == CANONICAL_TOKENIZER_SHA256

print("Canonical tokenizer decision: ACCEPTED")
print(f"  vocabulary: {canonical_config.vocab_size:,}")
print(f"  tokenizer version: {canonical_metadata.tokenizer_library_version}")
print(f"  tokenizer SHA-256: {canonical_metadata.tokenizer_sha256}")
print(f"  manifest SHA-256: {canonical_metadata.dataset_manifest_sha256}")
print(f"  accepted limitation: {canonical_metadata.accepted_limitations[0]}")

Canonical tokenizer decision: ACCEPTED
  vocabulary: 37,000
  tokenizer version: 0.23.1
  tokenizer SHA-256: 2826114029ded109107969c689cd62307e128dff84ad661211a26e4dfd272f74
  manifest SHA-256: 7b101e47d2e756fff61b8cb9a3d9c66228f148b02b7bb0d1d9047df9176fa66f
  accepted limitation: One <unk> token occurs across the 3,000 English development sentences.


## Goal

Learn a shared English-German BPE vocabulary and give it a stable identity.


## Paper and project contract

*Attention Is All You Need* reports a shared English-German BPE vocabulary of about 37,000 tokens and later shares the source embedding, target embedding, and pre-softmax weight matrix. This project keeps one shared vocabulary but may scale its size after measuring vocabulary use and sequence-length inflation. Training text must come only from the approved training split exposed by `iter_bpe_training_text`; development and final-test text must not influence the learned vocabulary.

The tokenizer identity will ultimately bind the training configuration, ordered vocabulary and merges, tokenizer-library version, and source dataset-manifest identity.


## Required deliverables

- Typed tokenizer API
- Fixture-trained BPE artifact
- Round-trip tests
- Vocabulary and sequence-length report


## Planned implementation sections

1. Paper and contract references
2. Typed implementation
3. Focused tests
4. Deterministic visible result
5. Exported API and artifact identities


## Explicitly deferred

Embeddings, positions, attention, and neural model code.


## HITL checkpoint

Maintainer decision: **approved**. The 37,000-token shared vocabulary, fixed special-token IDs, deterministic round trips, canonical artifact identity, development evidence, and the bounded single-`<unk>` limitation are accepted for issue #3.